# Post-processing of Textract output

Cleans the raw Textract CSV output of the 1956 CNDD tables to a form
suitable for cell-by-cell comparison with the manual ground truth.
Implements the post-processing steps described in Section 2.2 of the
paper.

**Pipeline (applied to both Textract and manual CSVs):**

1. Remove preamble rows that precede the first data row (Textract only;
   the manual CSVs have no preamble).
2. Remove the first two columns (row index and disease label columns,
   which are not part of the cell comparison).
3. Pad short rows with a placeholder so all rows in a file share the same
   column count.

**Inputs**
- `data/textract_raw/1956/cdi_ca_1956_wk_prov_dbs_Part{i}_tables.csv`
- `data/textract_raw/1956/cdi_ca_1956_wk_prov_dbs_Part{i}_confidences.csv` *(optional)*
- `data/manual/1956/cdi_ca_1956_wk_prov_dbs_Part{i}.csv`

**Outputs**
- `data/textract_cleaned/1956/cdi_ca_1956_wk_prov_dbs_Part{i}_tables.csv`
- `data/textract_cleaned/1956/cdi_ca_1956_wk_prov_dbs_Part{i}_confidences.csv` *(if input present)*
- `data/manual_cleaned/1956/cdi_ca_1956_wk_prov_dbs_Part{i}.csv`

Outputs are written in the form expected by `postprocessing/combine.ipynb`,
which builds the cell-comparison table used in Section 5.1.


In [ ]:
# Configuration. All paths are relative to the repository root.
# Run this notebook from the repository root, or adjust REPO_ROOT below.

import os

REPO_ROOT = ".."   # this notebook lives in postprocessing/

YEAR = 1956
N_WEEKS = 52
FILE_PREFIX = f"cdi_ca_{YEAR}_wk_prov_dbs_Part"

TEXTRACT_INPUT_DIR  = os.path.join(REPO_ROOT, "data", "textract_raw",     str(YEAR))
MANUAL_INPUT_DIR    = os.path.join(REPO_ROOT, "data", "manual",           str(YEAR))
TEXTRACT_OUTPUT_DIR = os.path.join(REPO_ROOT, "data", "textract_cleaned", str(YEAR))
MANUAL_OUTPUT_DIR   = os.path.join(REPO_ROOT, "data", "manual_cleaned",   str(YEAR))

# Textract output begins with a preamble (column headers, footnotes, etc).
# The first data row begins with the value "1" (the row index in the source PDF).
FIRST_DATA_ROW_MARKER = "1"

# Placeholder used when padding short rows to the file's maximum width.
PAD_PLACEHOLDER = "FILLER"


In [ ]:
# Cleaning helpers.

import csv

def count_rows_until_value(file_path, stop_value):
    """Return the number of rows preceding the first row whose first cell
    equals stop_value. If no such row exists, return 0 (do not trim)."""
    with open(file_path, "r", newline="", encoding="utf-8-sig") as f:
        for i, row in enumerate(csv.reader(f)):
            if row and row[0] == stop_value:
                return i
    return 0


def remove_first_n_rows(input_path, output_path, n):
    """Copy input_path to output_path, dropping the first n rows."""
    with open(input_path, "r", newline="", encoding="utf-8-sig") as fin:
        rows = list(csv.reader(fin))[n:]
    with open(output_path, "w", newline="", encoding="utf-8") as fout:
        csv.writer(fout).writerows(rows)


def remove_first_n_columns(input_path, output_path, n):
    """Copy input_path to output_path, dropping the first n columns of every row."""
    with open(input_path, "r", newline="", encoding="utf-8-sig") as fin:
        rows = [row[n:] for row in csv.reader(fin)]
    with open(output_path, "w", newline="", encoding="utf-8") as fout:
        csv.writer(fout).writerows(rows)


def pad_rows(input_path, output_path, placeholder=PAD_PLACEHOLDER):
    """Copy input_path to output_path, padding short rows with placeholder
    so every row has the same number of columns as the file's widest row."""
    with open(input_path, "r", newline="", encoding="utf-8-sig") as fin:
        rows = list(csv.reader(fin))
    max_cols = max((len(r) for r in rows), default=0)
    with open(output_path, "w", newline="", encoding="utf-8") as fout:
        writer = csv.writer(fout)
        for row in rows:
            if len(row) < max_cols:
                row = row + [placeholder] * (max_cols - len(row))
            writer.writerow(row)


In [ ]:
# Clean one Textract CSV (preamble removal, column trim, padding).

import tempfile

def clean_textract_file(src, dst):
    """Run the full Textract cleaning pipeline on a single CSV."""
    n_preamble = count_rows_until_value(src, FIRST_DATA_ROW_MARKER)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w") as tmp1, \
         tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w") as tmp2:
        tmp1_path, tmp2_path = tmp1.name, tmp2.name
    try:
        remove_first_n_rows(src, tmp1_path, n_preamble)
        remove_first_n_columns(tmp1_path, tmp2_path, n=2)
        pad_rows(tmp2_path, dst)
    finally:
        os.remove(tmp1_path)
        os.remove(tmp2_path)


def clean_manual_file(src, dst):
    """Run the manual-CSV cleaning pipeline (column trim, padding; no preamble)."""
    with tempfile.NamedTemporaryFile(delete=False, suffix=".csv", mode="w") as tmp:
        tmp_path = tmp.name
    try:
        remove_first_n_columns(src, tmp_path, n=2)
        pad_rows(tmp_path, dst)
    finally:
        os.remove(tmp_path)


In [ ]:
# Run the cleaning pipeline for the full year.

os.makedirs(TEXTRACT_OUTPUT_DIR, exist_ok=True)
os.makedirs(MANUAL_OUTPUT_DIR,   exist_ok=True)

processed = 0
skipped = []

for i in range(1, N_WEEKS + 1):
    base = f"{FILE_PREFIX}{i}"

    # Textract: tables (required)
    t_src = os.path.join(TEXTRACT_INPUT_DIR,  f"{base}_tables.csv")
    t_dst = os.path.join(TEXTRACT_OUTPUT_DIR, f"{base}_tables.csv")
    if os.path.exists(t_src):
        clean_textract_file(t_src, t_dst)
    else:
        skipped.append(t_src)

    # Textract: confidences (optional; the paper's figures don't use them)
    c_src = os.path.join(TEXTRACT_INPUT_DIR,  f"{base}_confidences.csv")
    c_dst = os.path.join(TEXTRACT_OUTPUT_DIR, f"{base}_confidences.csv")
    if os.path.exists(c_src):
        clean_textract_file(c_src, c_dst)

    # Manual
    m_src = os.path.join(MANUAL_INPUT_DIR,  f"{base}.csv")
    m_dst = os.path.join(MANUAL_OUTPUT_DIR, f"{base}.csv")
    if os.path.exists(m_src):
        clean_manual_file(m_src, m_dst)
    else:
        skipped.append(m_src)

    processed += 1

print(f"Processed {processed} weeks.")
print(f"Output:")
print(f"  {TEXTRACT_OUTPUT_DIR}/")
print(f"  {MANUAL_OUTPUT_DIR}/")
if skipped:
    print(f"\nMissing inputs ({len(skipped)}):")
    for s in skipped[:5]:
        print(f"  {s}")
    if len(skipped) > 5:
        print(f"  ... and {len(skipped) - 5} more")
